# Notebook 07: robustness — walk-forward, ablations, case study, Aviles 2024 (Section 10.2)

Runs on the chosen laptop **after Notebook 06 (G18)**. Nothing here changes a frozen setting or the H3/H4 verdicts.
Same as `python scripts/run_phase.py --phase 7` (each stage resumable: `--stage walkforward|ablations|casestudy|aviles`).
Seeds per G15 (`seeds_wf_ablation`), determinism on (D15).

* **Walk-forward** (2018–20 → 2021, 2018–21 → 2022, 2018–22 → 2023): per fold the tide, alert levels, Granger
  selection, training medians and scalers come from the fold's training years; the network trains on all but the
  last training year, which is the inner validation (early stopping, ridge alpha, trigger offsets). The 2023 fold
  is reported separately: 2023 chose the hyperparameters, so it is not independent.
* **Ablations** (Table III-5): no Granger selection, correlation selection, no rainfall, no tabular path, second
  LSTM instead of the GRU, weighted loss (D20). Each with its own trigger offsets on 2023, scored on the 2024 hours.
* **Case study**: every 2024 flood-risk event, including 24 Jul 2024 (Carina).
* **Aviles 2024**: descriptive check of the alert levels against the June–15 Jul 2024 records.

Dry run: `PASIG_PROFILE=rehearsal` (2023 stands in for 2024; numbers mean nothing).

**Runs locally:** the raw data are committed in `data/raw/`; the first cell checks them against the manifest and, in a fresh clone, rebuilds any missing generated inputs (no Google Drive needed).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from pasig.local import prepare
prepare(7)   # raw data from the repo (no Drive); rebuilds missing local inputs once per clone
# os.environ['PASIG_PROFILE'] = 'rehearsal'   # uncomment for the dry run
import pandas as pd
from IPython.display import Image, display
from pasig import phase7
from pasig.config import out_dir, load, profile
from pasig.gates import require_gates
require_gates('G18')
T, F = out_dir('tables'), out_dir('figures')
profile(), phase7.seeds()

## 1. Walk-forward

In [ ]:
assert phase7.run(['walkforward']) == 0
folds = json.loads((T / 'walkforward_folds.json').read_text())['folds']
pd.DataFrame(folds).T[['train_years', 'inner_validation_year', 'eval_year', 'high_water_m', 'flood_risk_m',
                       'granger_selected', 'tide_added', 'ridge_alpha', 'cases']]

In [ ]:
lv = pd.read_csv(T / 'walkforward_level_metrics.csv')
lv[lv.subset.eq('all')].pivot_table(index=['fold', 'model'], columns='lead', values='rmse').round(4)

In [ ]:
dm = pd.read_csv(T / 'walkforward_dm_holm.csv')
dm.pivot_table(index=['fold', 'reference'], columns='lead', values='p_holm').round(4)   # hybrid vs baseline; descriptive

In [ ]:
al = pd.read_csv(T / 'walkforward_alerts.csv')
al[al.subset.eq('all') & al.lead.eq(3)].pivot_table(index=['fold', 'model'], columns='level', values='f1').round(3)

## 2. Ablations (Table III-5) and the weighted loss (D20)

In [ ]:
assert phase7.run(['ablations']) == 0
d = pd.read_csv(T / 'ablation_rmse_diff.csv')
d.pivot_table(index='variant', columns='lead', values='rmse_diff').round(4)   # ablation - main hybrid (same seeds)

In [ ]:
d.pivot_table(index='variant', columns='lead', values='beyond_seed_noise', aggfunc='first')

In [ ]:
pd.read_csv(T / 'ablation_dm_holm.csv').pivot_table(index='model', columns='lead', values='p_holm').round(4)

In [ ]:
a = pd.read_csv(T / 'ablation_alerts.csv')
a[a.subset.eq('all')].pivot_table(index='model', columns=['level', 'lead'], values='f1').round(3)

In [ ]:
json.loads((T / 'ablation_d20.json').read_text())

## 3. Case study: every flood-risk event of the test season

In [ ]:
assert phase7.run(['casestudy']) == 0
pd.read_csv(T / 'case_study_events.csv')

In [ ]:
cs = pd.read_csv(T / 'case_study_detail.csv')
cs.pivot_table(index=['event', 'model'], columns='lead', values='warning_h').round(1)

In [ ]:
for p in sorted(F.glob('case_event_*.png')):
    display(Image(p))

## 4. Aviles 2024: descriptive check of the alert levels

In [ ]:
assert phase7.run(['aviles']) == 0
pd.read_csv(T / 'aviles_check.csv')

## Tests

In [ ]:
import subprocess
assert subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_same_hours.py', 'tests/test_walkforward.py']) == 0

Record every result in `reports/transparency_log.md` (beats/does not beat, as measured) and commit
`reports/tables/` (walk-forward, ablation, case-study, Aviles files) and `reports/figures/case_event_*.png`.